# Real search evaluation: can we safely change an index?

**Workflow:** a team maintains search over scientific abstracts. Someone proposes indexing titles only. Does the change preserve retrieval quality—and do we have enough evidence for every required cohort?

This notebook uses **all 5,183 documents and 300 test queries from BEIR SciFact**. It builds actual local TF-IDF rankings, evaluates the published binary relevance judgments, and turns a before/after comparison into a repeatable release gate.

No LLM, paid API, GPU, or model download. This is **retrieval of potentially relevant evidence**, not verification of scientific claims or medical advice.

### Start here

From a cloned checkout:
```bash
python -m pip install -e .
python -m pip install -r examples/requirements-notebooks.txt
python -m jupyterlab examples/notebooks
```
Select the kernel from that environment, then **Restart Kernel and Run All**. The data-loading cell explicitly downloads about 2.8 MB over HTTPS. Offline alternative: set `PROOFML_SCIFACT_ARCHIVE` to an already downloaded official ZIP before launching Jupyter.

The committed outputs are a recorded run, not fabricated examples. Re-running creates a new artifact directory rather than overwriting previous evidence.

In [1]:
from pathlib import Path
import os
import sys
import tempfile
import pandas as pd
from IPython.display import display

# Run from the repository root or examples/notebooks, with this environment's kernel.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "examples/scifact_retrieval.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Clone proofml and launch this notebook inside that checkout.")
sys.path.insert(0, str(ROOT))
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
from proofml import __version__
print("ProofML", __version__)

ProofML 0.7.1


## 1. Load a fixed, judged benchmark

The helper checks the archive and all three input-file SHA-256 hashes, validates IDs, and selects **only the BEIR test qrels**. There are more queries in the archive than judged test queries; silently evaluating all of them would change the population.

A *qrel* is a query–document relevance judgment. Here positive judgments have score 1. Unlisted documents are treated as nonrelevant for this benchmark, but may be genuinely relevant and unjudged.

[BEIR source](https://github.com/beir-cellar/beir) · [SciFact authors](https://github.com/allenai/scifact) · [Readable helper code](../scifact_retrieval.py)

In [2]:
from examples.scifact_retrieval import (
    load_scifact, query_groups, rank_tfidf, release_policy, metrics_table,
    save_run, ARCHIVE_SHA256, MEMBER_SHA256, K,
)
archive = os.environ.get("PROOFML_SCIFACT_ARCHIVE")
benchmark = load_scifact(archive=archive) if archive else load_scifact(download=True)
print({
    "documents": len(benchmark.corpus),
    "test_queries": len(benchmark.queries),
    "positive_judgments": sum(map(len, benchmark.relevant.values())),
})
print("Archive SHA-256:", ARCHIVE_SHA256)

{'documents': 5183, 'test_queries': 300, 'positive_judgments': 339}
Archive SHA-256: 536e14446a0ba56ed1398ab1055f39fe852686ecad24a6306c80c490fa8e0165


## 2. Define the experiment before looking at scores

Baseline: index **title + abstract**. Candidate: index **title only**. Both use unigram TF-IDF, sublinear term frequency, L2 normalization, cosine similarity, the same corpus IDs, and the same top-10 cutoff. Vocabulary/IDF are fit on the searchable corpus, never on query judgments. No test-set parameter search.

We define short queries as **at most 12 Unicode word tokens**, and long queries as more than 12, before evaluation. These are illustrative lexical cohorts—not demographic or clinical groups.

The policy requires every named cohort, complete positive-judgment coverage, at least 20 assessed queries per cohort, and no recall@10 or nDCG@10 decline larger than **0.05 absolute units**. These are teaching choices, not statistically calibrated guarantees. A production team should choose them before evaluating candidates.

In [3]:
groups = query_groups(benchmark.queries)
policy = release_policy()
display(pd.Series(groups).value_counts().rename("queries").to_frame())
print("Required reports:", policy.require_reports)

,queries
short_queries,163
long_queries,137


Required reports: ('overall', 'short_queries', 'long_queries')


## 3. Retrieve for real

The ranking helper contains the complete retrieval recipe. It scores one query at a time, avoiding a dense all-query-by-corpus matrix. Equal scores break ties by document ID; zero-score matches are omitted rather than padded.

This compares index content, **not measured latency or storage savings**. TF-IDF is a transparent educational baseline, not BEIR's BM25 or a published leaderboard run.

In [4]:
baseline_rankings = rank_tfidf(benchmark.corpus, benchmark.queries, fields=("title", "text"))
candidate_rankings = rank_tfidf(benchmark.corpus, benchmark.queries, fields=("title",))
assert set(baseline_rankings) == set(candidate_rankings) == set(benchmark.relevant)
print("Two real ranking runs:", len(baseline_rankings), "queries each")

Two real ranking runs: 300 queries each


## 4. Audit with a few lines

This is the reusable ProofML integration. Any search engine can supply the same `{query_id: [ranked_document_ids]}` mapping. The evaluator does not need to own your index or call your retriever.

Keeping stable query IDs, qrels, and corpus IDs makes comparison possible. These hashes do **not** prove unchanged query/document text; the separately recorded input-file hashes supply that provenance here. In your own system, version content and case IDs.

In [5]:
from proofml import audit_retrieval_slices

audit_options = dict(groups=groups, corpus_ids=tuple(benchmark.corpus), k=K)
baseline = audit_retrieval_slices(baseline_rankings, benchmark.relevant, **audit_options)
candidate = audit_retrieval_slices(candidate_rankings, benchmark.relevant, **audit_options)
decision = policy.evaluate(candidate, baseline=baseline)
display(metrics_table(baseline, candidate, decision).round(4))
print("Release decision:", decision.status)

,cohort,queries,baseline_recall@10,candidate_recall@10,baseline_ndcg@10,candidate_ndcg@10,gate
0,overall,300,0.7735,0.5476,0.6286,0.4166,failed
1,short_queries,163,0.7727,0.5322,0.6385,0.4094,failed
2,long_queries,137,0.7745,0.5659,0.6167,0.4251,failed


Release decision: failed


**Read the table:** recall@10 measures the fraction of known relevant documents retrieved. nDCG@10 additionally rewards placing them earlier. Both are macro-averaged over queries with positive judgments. Each cohort is assessed separately; counts are not pooled into another average.

The table above is the measured result, not an assertion that title-only retrieval must always fail. This experiment can reveal regression even when a few queries improve. It does not demonstrate that an improved aggregate masks a cohort decline unless the actual numbers show that.

In [6]:
# Independent recall calculation: validate the integration, not just the display.
import math
def reference_recall(rankings):
    return math.fsum(len(set(rankings[q][:K]) & benchmark.relevant[q]) /
                     len(benchmark.relevant[q]) for q in sorted(benchmark.relevant)) / len(benchmark.relevant)

for suite, rankings in ((baseline, baseline_rankings), (candidate, candidate_rankings)):
    measured = suite.metrics["overall"]["retrieval_ranking"]["recall@10"]
    assert math.isclose(measured, reference_recall(rankings), abs_tol=1e-12)
print("Both recall values match an independent direct calculation.")

Both recall values match an independent direct calculation.


## 5. Inspect why a gate blocks, then stop the pipeline

Use `policy.enforce(candidate, baseline=baseline)` in CI before deploying. It raises `AuditFailed` on regression or insufficient/incompatible evidence. We catch that expected outcome here so **Run All** completes and teaches the failure path.

The unchanged baseline is a useful passing control. A pass is compliance with the configured policy, not certification of the search system.

In [7]:
from proofml import AuditFailed
assert policy.evaluate(baseline, baseline=baseline).passed
for name, result in decision.results.items():
    for issue in result.issues:
        print(name, issue.code, issue.message)
try:
    policy.enforce(candidate, baseline=baseline)
except AuditFailed as error:
    print("Deployment blocked:", error.decision.status)
else:
    print("Candidate met the declared policy.")

overall metric_regression retrieval_ranking.recall@10 changed from 0.7735 to 0.547611, exceeding max_drop tolerance 0.05.
overall metric_regression retrieval_ranking.ndcg@10 changed from 0.628553 to 0.41658, exceeding max_drop tolerance 0.05.
long_queries metric_regression retrieval_ranking.recall@10 changed from 0.774453 to 0.565937, exceeding max_drop tolerance 0.05.
long_queries metric_regression retrieval_ranking.ndcg@10 changed from 0.616662 to 0.425146, exceeding max_drop tolerance 0.05.
short_queries metric_regression retrieval_ranking.recall@10 changed from 0.772699 to 0.532209, exceeding max_drop tolerance 0.05.
short_queries metric_regression retrieval_ranking.ndcg@10 changed from 0.638547 to 0.409381, exceeding max_drop tolerance 0.05.
Deployment blocked: failed


## 6. Two common integration mistakes

The next two cells are **deliberately injected faults**, separate from the real title-only comparison:

1. An exporter changes the document-ID namespace without updating the corpus/qrels.
2. A dashboard accidentally drops the long-query cohort.

A score alone cannot explain the first problem; a passing subset must not conceal the second.

In [8]:
from proofml import audit_retrieval, AuditPolicy

namespace_fault = {q: ["stale::" + doc for doc in ranked] for q, ranked in candidate_rankings.items()}
fault_report = audit_retrieval(namespace_fault, benchmark.relevant, corpus_ids=tuple(benchmark.corpus), k=K)
assert any(f.code == "unknown_retrieved_ids" for f in fault_report.findings)
fault_decision = AuditPolicy(require_checks=("retrieval_corpus", "retrieval_ranking")).evaluate(fault_report)
print("Injected namespace fault:", fault_decision.status)
print("Findings:", sorted({f.code for f in fault_report.findings}))

Injected namespace fault: failed
Findings: ['unknown_retrieved_ids']


In [9]:
from proofml import AuditSuite

missing_cohort = AuditSuite({name: report for name, report in candidate.reports.items()
                            if name != "long_queries"})
missing_decision = policy.evaluate(missing_cohort, baseline=baseline)
assert missing_decision.status == "insufficient"
print("Missing-cohort gate:", missing_decision.status)
for issue in missing_decision.issues:
    print(issue.code, issue.message)

Missing-cohort gate: insufficient
required_report_missing Required report long_queries is absent from the candidate.
candidate_report_removed Baseline report long_queries is absent from the candidate.


## 7. Save evidence and run the same policy in CI

Reports retain measurements, compatibility fingerprints, coverage, and findings. Decisions retain the rule evidence. JUnit gives CI systems a test case for each cohort.

Saved reports omit query/document text. Corpus IDs and query IDs are hashed by ProofML, but hashes and small-population aggregates can still be sensitive. Treat your own artifacts according to your data policy.

In [10]:
RUN_DIR = Path(tempfile.mkdtemp(prefix="scifact-notebook-", dir=ARTIFACTS)) / "evidence"
recorded = save_run(RUN_DIR, baseline, candidate, decision)
print("Saved reports:", RUN_DIR.relative_to(ROOT).as_posix())
print("Replay in CI:")
print(f"proofml gate {RUN_DIR.relative_to(ROOT)}/candidate/suite.json --suite "
      f"--policy {RUN_DIR.relative_to(ROOT)}/policy.json "
      f"--baseline {RUN_DIR.relative_to(ROOT)}/baseline/suite.json")
from proofml import AuditSuite, SuitePolicy
reloaded = SuitePolicy.from_file(RUN_DIR / "policy.json").evaluate(
    AuditSuite.load(RUN_DIR / "candidate/suite.json"),
    baseline=AuditSuite.load(RUN_DIR / "baseline/suite.json"),
)
assert reloaded.status == decision.status
print("Saved-policy replay agrees:", reloaded.status)

Saved reports: artifacts/scifact-notebook-r6che7u0/evidence
Replay in CI:
proofml gate artifacts/scifact-notebook-r6che7u0/evidence/candidate/suite.json --suite --policy artifacts/scifact-notebook-r6che7u0/evidence/policy.json --baseline artifacts/scifact-notebook-r6che7u0/evidence/baseline/suite.json
Saved-policy replay agrees: failed


## Apply this in your workflow

Replace the two TF-IDF calls with exported results from Elasticsearch, OpenSearch, a vector database, or your RAG retriever. Keep the same stable IDs, relevance judgments, cohorts, and policy. Run it when changing chunking, indexing fields, filters, or reranking.

**Exercises:** add an explicitly defined query-type cohort; check an empty required cohort; compare another untuned retriever while holding the benchmark fixed. Do not tune repeatedly on these test judgments and then call the resulting score an untouched test estimate.

**Boundaries:** positive-only judgment coverage does not mean exhaustive relevance coverage; query length is not fairness analysis; five percentage points is not a significance test; relevant scientific evidence is not necessarily supporting evidence. There is no answer-generation or factual-correctness evaluation here.

### Sources and attribution

- Thakur et al., *BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models* (2021), [benchmark repository](https://github.com/beir-cellar/beir).
- Wadden et al., *Fact or Fiction: Verifying Scientific Claims* (2020), [SciFact repository](https://github.com/allenai/scifact).
- [Upstream component license notice](https://github.com/allenai/scifact/blob/master/LICENSE.md): claims/evidence annotations CC BY 4.0; corpus abstracts ODC-By 1.0. Dataset rights are separate from ProofML's MIT code license. Raw text is downloaded at execution time and is not committed here.
- [TF-IDF implementation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html). These results are our recipe, not a claimed reproduction of a BEIR leaderboard model.